# GRU model selection — BRGM piezometric datasets

This notebook contains **only the GRU forecasting/model-selection code** used to produce the saved BRGM forecasters for the paper.


## 1. Imports and GRU-specific configuration


In [ ]:
from __future__ import annotations

import random
import warnings
import zlib
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

DEFAULT_DATASETS: Tuple[str, ...] = (
    "piezo1", "piezo2", "piezo3", "piezo4", "piezo5", "piezo6",
    "piezo7", "piezo8", "piezo9", "piezo11", "piezo12", "piezo13",
    "piezo14", "piezo15", "piezo16", "piezo17", "piezo18",
)

@dataclass
class ExperimentConfig:
    datasets: Tuple[str, ...] = DEFAULT_DATASETS
    input_len: int = 336
    horizon: int = 24
    target_col: str = "level"
    train_ratio: float = 0.5
    val_ratio: float = 0.1
    batch_size: int = 128
    gru_hidden: int = 100
    gru_layers: int = 2
    gru_lrs: Tuple[float, ...] = (1e-4, 3e-4, 1e-5, 3e-5, 1e-3, 3e-3)
    gru_epochs: int = 80
    patience: int = 10
    change_feature_names: Tuple[str, ...] = ("PRELIQ_Q", "T_Q", "ETP_Q")
    eval_windows: int = 15
    example_plots_per_dataset: int = 1
    num_workers: int = 0
    seed: int = 42
    device: str = "cpu"


## 2. GRU data, training, evaluation, and plotting utilities


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def safe_mkdir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def stable_seed(base_seed: int, name: str) -> int:
    return int((base_seed + zlib.crc32(name.encode("utf-8"))) % (2**32 - 1))

def format_lr(value: float) -> str:
    return f"{value:.0e}" if value < 0.001 else f"{value:g}"

class DatasetScan:
    name: str
    csv_path: Path
    n_rows: int
    test_windows: int

class TrainingResult:
    lr: float
    model: nn.Module
    history: Dict[str, List[float]]
    best_val_loss: float
    val_metrics: Dict[str, float]

class SlidingWindowDataset(Dataset):
    """Notebook-faithful window construction without materializing all samples."""

    def __init__(
        self,
        full_scaled: np.ndarray,
        start: int,
        end: int,
        input_len: int,
        horizon: int,
        target_idx: int,
    ) -> None:
        self.full_scaled = full_scaled
        self.start = start
        self.end = end
        self.input_len = input_len
        self.horizon = horizon
        self.target_idx = target_idx
        self.segment_len = max(end - start, 0)
        # Preserves the notebook's range(len(segment) - input_len - horizon).
        self.num_samples = max(self.segment_len - input_len - horizon, 0)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, int]:
        start_idx = self.start + idx
        x = self.full_scaled[start_idx : start_idx + self.input_len]
        y = self.full_scaled[
            start_idx + self.input_len : start_idx + self.input_len + self.horizon,
            self.target_idx,
        ]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            start_idx,
        )

class PreparedDataset:
    name: str
    csv_path: Path
    df: pd.DataFrame
    feature_cols: List[str]
    target_idx: int
    change_feature_idx: List[int]
    static_feature_idx: List[int]
    scaler: StandardScaler
    values_scaled: np.ndarray
    train_range: Tuple[int, int]
    val_range: Tuple[int, int]
    test_range: Tuple[int, int]
    train_ds: SlidingWindowDataset
    val_ds: SlidingWindowDataset
    test_ds: SlidingWindowDataset

    def inverse_target(self, y_scaled_1d: np.ndarray) -> np.ndarray:
        tmp = np.zeros((len(y_scaled_1d), len(self.feature_cols)), dtype=np.float32)
        tmp[:, self.target_idx] = y_scaled_1d
        inv = self.scaler.inverse_transform(tmp)
        return inv[:, self.target_idx]

    def inverse_input(self, x_scaled_2d: np.ndarray) -> np.ndarray:
        return self.scaler.inverse_transform(x_scaled_2d)

    def make_loader(
        self,
        split: str,
        batch_size: int,
        shuffle: bool,
        seed: int,
        num_workers: int,
    ) -> DataLoader:
        dataset = {
            "train": self.train_ds,
            "val": self.val_ds,
            "test": self.test_ds,
        }[split]
        generator = None
        if shuffle:
            generator = torch.Generator()
            generator.manual_seed(seed)
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            generator=generator,
            num_workers=num_workers,
        )

class GRUForecaster(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, horizon: int):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0,
        )
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

def discover_dataset_files(search_root: Path, datasets: Sequence[str]) -> Tuple[List[Path], List[str]]:
    requested = {name.lower(): name for name in datasets}
    found: Dict[str, Path] = {}
    for csv_path in search_root.rglob("*.csv"):
        stem = csv_path.stem.lower()
        if stem in requested and stem not in found:
            found[stem] = csv_path
    ordered_paths = [found[name.lower()] for name in datasets if name.lower() in found]
    missing = [name for name in datasets if name.lower() not in found]
    return ordered_paths, missing

def load_piezo_dataframe(csv_path: Path, cfg: ExperimentConfig) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required_cols = {"date", cfg.target_col, *cfg.change_feature_names}
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(
            f"{csv_path.name} is missing required columns: {', '.join(missing_cols)}"
        )
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)

def compute_window_counts(n_rows: int, cfg: ExperimentConfig) -> Tuple[int, int, int, int, int]:
    n_train = int(n_rows * cfg.train_ratio)
    n_val = int(n_rows * cfg.val_ratio)
    train_start, train_end = 0, n_train
    val_start, val_end = n_train - cfg.input_len - cfg.horizon, n_train + n_val
    test_start, test_end = n_train + n_val - cfg.input_len - cfg.horizon, n_rows
    test_windows = max((test_end - test_start) - cfg.input_len - cfg.horizon, 0)
    return n_train, n_val, train_start, val_start, test_windows

def scan_dataset(csv_path: Path, cfg: ExperimentConfig) -> DatasetScan:
    df = load_piezo_dataframe(csv_path, cfg)
    n_train, _, _, _, test_windows = compute_window_counts(len(df), cfg)
    return DatasetScan(
        name=csv_path.stem,
        csv_path=csv_path,
        n_rows=len(df),
        test_windows=test_windows,
    )

def prepare_dataset(csv_path: Path, cfg: ExperimentConfig) -> PreparedDataset:
    df = load_piezo_dataframe(csv_path, cfg)
    feature_cols = [col for col in df.columns if col != "date"]
    target_idx = feature_cols.index(cfg.target_col)
    change_feature_idx = [feature_cols.index(col) for col in cfg.change_feature_names]
    static_feature_idx = [idx for idx in range(len(feature_cols)) if idx not in change_feature_idx]

    values = df[feature_cols].values.astype(np.float32)
    n_rows = len(values)
    n_train = int(n_rows * cfg.train_ratio)
    n_val = int(n_rows * cfg.val_ratio)

    scaler = StandardScaler()
    scaler.fit(values[:n_train])
    values_scaled = scaler.transform(values).astype(np.float32)

    train_range = (0, n_train)
    val_range = (n_train - cfg.input_len - cfg.horizon, n_train + n_val)
    test_range = (n_train + n_val - cfg.input_len - cfg.horizon, n_rows)

    train_ds = SlidingWindowDataset(values_scaled, *train_range, cfg.input_len, cfg.horizon, target_idx)
    val_ds = SlidingWindowDataset(values_scaled, *val_range, cfg.input_len, cfg.horizon, target_idx)
    test_ds = SlidingWindowDataset(values_scaled, *test_range, cfg.input_len, cfg.horizon, target_idx)

    if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
        raise ValueError(
            f"{csv_path.name} is too short for input_len={cfg.input_len} and horizon={cfg.horizon}."
        )

    return PreparedDataset(
        name=csv_path.stem,
        csv_path=csv_path,
        df=df,
        feature_cols=feature_cols,
        target_idx=target_idx,
        change_feature_idx=change_feature_idx,
        static_feature_idx=static_feature_idx,
        scaler=scaler,
        values_scaled=values_scaled,
        train_range=train_range,
        val_range=val_range,
        test_range=test_range,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
    )

def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: Optional[torch.optim.Optimizer] = None,
) -> float:
    train_mode = optimizer is not None
    model.train(train_mode)
    total = 0.0
    n = 0
    for xb, yb, _ in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        if train_mode:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total += loss.item() * xb.size(0)
        n += xb.size(0)
    return total / max(n, 1)

def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    preds: List[np.ndarray] = []
    ys: List[np.ndarray] = []
    starts: List[np.ndarray] = []
    xs: List[np.ndarray] = []
    for xb, yb, start_idx in loader:
        pred = model(xb.to(device)).cpu().numpy()
        preds.append(pred)
        ys.append(yb.numpy())
        starts.append(start_idx.numpy())
        xs.append(xb.numpy())
    return (
        np.concatenate(xs),
        np.concatenate(ys),
        np.concatenate(preds),
        np.concatenate(starts),
    )

def compute_regression_metrics(y_true_inv: np.ndarray, y_pred_inv: np.ndarray) -> Dict[str, float]:
    metrics = {
        "mae": float(mean_absolute_error(y_true_inv.ravel(), y_pred_inv.ravel())),
        "rmse": float(np.sqrt(mean_squared_error(y_true_inv.ravel(), y_pred_inv.ravel()))),
    }
    try:
        metrics["r2"] = float(r2_score(y_true_inv.ravel(), y_pred_inv.ravel()))
    except Exception:
        metrics["r2"] = float("nan")
    return metrics

def train_gru(bundle: PreparedDataset, cfg: ExperimentConfig, lr: float) -> TrainingResult:
    set_seed(cfg.seed)
    model = GRUForecaster(
        input_size=len(bundle.feature_cols),
        hidden_size=cfg.gru_hidden,
        num_layers=cfg.gru_layers,
        horizon=cfg.horizon,
    ).to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    train_loader = bundle.make_loader("train", cfg.batch_size, True, cfg.seed, cfg.num_workers)
    val_loader = bundle.make_loader("val", cfg.batch_size, False, cfg.seed, cfg.num_workers)

    best_state = None
    best_val = float("inf")
    wait = 0
    history = {"train": [], "val": []}

    for epoch in range(1, cfg.gru_epochs + 1):
        tr = run_epoch(model, train_loader, criterion, cfg.device, optimizer)
        va = run_epoch(model, val_loader, criterion, cfg.device, optimizer=None)
        history["train"].append(tr)
        history["val"].append(va)
        print(f"    GRU lr={lr:g} | epoch {epoch:03d} | train={tr:.6f} | val={va:.6f}")
        if va < best_val - 1e-5:
            best_val = va
            best_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= cfg.patience:
                print("    Early stopping.")
                break

    if best_state is None:
        raise RuntimeError(f"Training failed for {bundle.name} at lr={lr}.")
    model.load_state_dict(best_state)

    x_val, y_val, y_val_pred, _ = predict_loader(model, val_loader, cfg.device)
    y_val_inv = np.stack([bundle.inverse_target(row) for row in y_val])
    y_val_pred_inv = np.stack([bundle.inverse_target(row) for row in y_val_pred])
    val_metrics = compute_regression_metrics(y_val_inv, y_val_pred_inv)
    val_metrics["best_val_loss"] = float(best_val)
    val_metrics["n_val_windows"] = int(len(x_val))

    return TrainingResult(
        lr=lr,
        model=model,
        history=history,
        best_val_loss=float(best_val),
        val_metrics=val_metrics,
    )

def pick_best_training_result(results: Sequence[TrainingResult]) -> TrainingResult:
    return min(
        results,
        key=lambda res: (
            res.val_metrics["mae"],
            res.val_metrics["rmse"],
            res.best_val_loss,
        ),
    )

def select_eval_indices(dataset_name: str, n_windows: int, eval_windows: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(stable_seed(seed, dataset_name))
    selected = rng.choice(n_windows, size=eval_windows, replace=False)
    return np.sort(selected.astype(int))

def plot_training_curve(history: Dict[str, List[float]], title: str, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["train"], label="train")
    ax.plot(history["val"], label="val")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MAE loss")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

def plot_lr_search(dataset_name: str, grid_df: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ordered = grid_df.sort_values("gru_lr")
    ax.plot(range(len(ordered)), ordered["val_mae"], marker="o", linewidth=2)
    ax.set_xticks(range(len(ordered)))
    ax.set_xticklabels([format_lr(value) for value in ordered["gru_lr"]], rotation=45)
    ax.set_xlabel("GRU learning rate")
    ax.set_ylabel("Validation MAE")
    ax.set_title(f"{dataset_name} GRU LR grid search")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

def plot_forecast_example(
    bundle: PreparedDataset,
    cfg: ExperimentConfig,
    x_scaled: np.ndarray,
    y_true_scaled: np.ndarray,
    y_pred_scaled: np.ndarray,
    start_idx: int,
    title: str,
    output_path: Path,
) -> None:
    x_inv = bundle.inverse_input(x_scaled)
    y_true_inv = bundle.inverse_target(y_true_scaled)
    y_pred_inv = bundle.inverse_target(y_pred_scaled)
    hist_dates = bundle.df["date"].iloc[start_idx : start_idx + cfg.input_len]
    fut_dates = bundle.df["date"].iloc[
        start_idx + cfg.input_len : start_idx + cfg.input_len + cfg.horizon
    ]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(hist_dates, x_inv[:, bundle.target_idx], label="history (target)")
    ax.plot(fut_dates, y_true_inv, label="true future")
    ax.plot(fut_dates, y_pred_inv, "--", label="predicted future")
    ax.axvline(hist_dates.iloc[-1], color="gray", linestyle="--")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

def plot_heatmap(
    data: pd.DataFrame,
    title: str,
    output_path: Path,
    cmap: str = "viridis",
) -> None:
    fig, ax = plt.subplots(figsize=(max(10, 0.7 * len(data.columns)), max(3, 0.5 * len(data.index) + 2)))
    im = ax.imshow(data.values, aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(data.columns)))
    ax.set_xticklabels(list(data.columns), rotation=45, ha="right")
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(list(data.index))
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

def plot_forecast_metrics_overview(forecast_df: pd.DataFrame, output_path: Path) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, metric in zip(axes, ["test_mae_full", "test_rmse_full", "test_r2_full"]):
        ax.bar(forecast_df["dataset"], forecast_df[metric])
        ax.set_title(metric.replace("_", " ").upper())
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)



## 3. Repository paths and exact experiment settings


In [ ]:
def find_project_root(start=None):
    """Locate the repository root from the current notebook working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from somewhere "
        "inside CFE_for_MTS_Forecasting_Framework."
    )

PROJECT_ROOT = find_project_root()
SEARCH_ROOT = PROJECT_ROOT / "Data" / "piezos"
OUTPUT_DIR = PROJECT_ROOT / "Results" / "EDBT_piezo_GRU"

DATASETS = [
    "piezo1", "piezo2", "piezo3", "piezo4", "piezo5", "piezo6",
    "piezo7", "piezo8", "piezo9", "piezo11", "piezo12", "piezo13",
    "piezo14", "piezo15", "piezo16", "piezo17", "piezo18",
]

GRU_LRS = [1e-4, 3e-4, 1e-5, 3e-5, 1e-3, 3e-3]
EVAL_WINDOWS = 15
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 128
GRU_EPOCHS = 80
EXAMPLE_PLOTS_PER_DATASET = 1
NUM_WORKERS = 0

cfg = ExperimentConfig(
    datasets=tuple(DATASETS),
    gru_lrs=tuple(GRU_LRS),
    eval_windows=EVAL_WINDOWS,
    batch_size=BATCH_SIZE,
    gru_epochs=GRU_EPOCHS,
    example_plots_per_dataset=EXAMPLE_PLOTS_PER_DATASET,
    num_workers=NUM_WORKERS,
    seed=SEED,
    device=DEVICE,
)

print("Project root:", PROJECT_ROOT)
print("Data:", SEARCH_ROOT)
print("Output:", OUTPUT_DIR)
print("Device:", DEVICE)
cfg


## 4. Verify the 17 input datasets


In [ ]:
found_paths, missing_datasets = discover_dataset_files(
    SEARCH_ROOT.resolve(),
    cfg.datasets,
)

scan_df = pd.DataFrame(
    {
        "dataset": [path.stem for path in found_paths],
        "csv_path": [str(path) for path in found_paths],
    }
)

display(scan_df)
print("Missing datasets:", missing_datasets if missing_datasets else "None")


## 5. Train and select the GRU forecaster for each piezometer

For each dataset, the notebook evaluates the same learning-rate grid, selects the model by validation MAE (with RMSE and validation loss as tie-breakers), evaluates the selected model on the full test set, and saves the checkpoint and result tables.


In [ ]:
def run_gru_model_selection(
    search_root: Path,
    output_dir: Path,
    cfg: ExperimentConfig,
) -> Path:
    """Train, select, evaluate, and save the BRGM GRU forecasters.

    This is the GRU-only subset of the original executed piezometer pipeline.
    The training loop, learning-rate selection, test evaluation, sampled-window
    selection, checkpoint saving, and plotting calls are preserved.
    """
    search_root = search_root.resolve()
    dataset_root = safe_mkdir(output_dir / "datasets")
    plots_root = safe_mkdir(output_dir / "plots")
    safe_mkdir(output_dir)

    found_paths, missing_datasets = discover_dataset_files(search_root, cfg.datasets)
    if not found_paths:
        raise FileNotFoundError(
            f"No requested piezo CSVs were found under {search_root}."
        )

    scans: List[DatasetScan] = []
    skipped: List[str] = []
    for csv_path in found_paths:
        try:
            scans.append(scan_dataset(csv_path, cfg))
        except Exception as exc:
            skipped.append(f"{csv_path.stem}: {exc}")

    if skipped:
        print("Skipped during scan:")
        for item in skipped:
            print(f"  - {item}")

    if not scans:
        raise RuntimeError("All discovered datasets failed the scan stage.")

    min_test_windows = min(scan.test_windows for scan in scans)
    common_eval_windows = min(cfg.eval_windows, min_test_windows)
    if common_eval_windows <= 0:
        raise RuntimeError("At least one dataset has zero available test windows.")

    print(f"Datasets processed: {[scan.name for scan in scans]}")
    print(f"Missing datasets: {missing_datasets if missing_datasets else 'None'}")
    print(f"Shared evaluation windows per dataset: {common_eval_windows}")

    all_grid_rows: List[Dict[str, object]] = []
    all_forecast_rows: List[Dict[str, object]] = []

    for scan in scans:
        print(f"\n=== Processing {scan.name} ({scan.csv_path}) ===")
        dataset_dir = safe_mkdir(dataset_root / scan.name)
        bundle = prepare_dataset(scan.csv_path, cfg)

        grid_results: List[TrainingResult] = []
        for lr in cfg.gru_lrs:
            training_result = train_gru(bundle, cfg, lr)
            grid_results.append(training_result)
            all_grid_rows.append(
                {
                    "dataset": scan.name,
                    "gru_lr": float(lr),
                    "best_val_loss": training_result.best_val_loss,
                    "val_mae": training_result.val_metrics["mae"],
                    "val_rmse": training_result.val_metrics["rmse"],
                    "val_r2": training_result.val_metrics["r2"],
                    "n_val_windows": training_result.val_metrics["n_val_windows"],
                }
            )

        best_result = pick_best_training_result(grid_results)
        torch.save(best_result.model.state_dict(), dataset_dir / "best_gru_forecaster.pth")

        plot_training_curve(
            best_result.history,
            title=f"{scan.name} GRU training curve (best lr={format_lr(best_result.lr)})",
            output_path=dataset_dir / "best_gru_training_curve.png",
        )

        dataset_grid_df = pd.DataFrame(
            [row for row in all_grid_rows if row["dataset"] == scan.name]
        )
        plot_lr_search(
            scan.name,
            dataset_grid_df,
            dataset_dir / "gru_lr_grid_search.png",
        )

        test_loader = bundle.make_loader(
            "test", cfg.batch_size, False, cfg.seed, cfg.num_workers
        )
        x_test, y_test, y_pred, test_starts = predict_loader(
            best_result.model, test_loader, cfg.device
        )
        y_test_inv = np.stack([bundle.inverse_target(row) for row in y_test])
        y_pred_inv = np.stack([bundle.inverse_target(row) for row in y_pred])

        full_test_metrics = compute_regression_metrics(y_test_inv, y_pred_inv)
        eval_indices = select_eval_indices(
            scan.name, len(x_test), common_eval_windows, cfg.seed
        )
        sampled_test_metrics = compute_regression_metrics(
            y_test_inv[eval_indices], y_pred_inv[eval_indices]
        )

        all_forecast_rows.append(
            {
                "dataset": scan.name,
                "csv_path": str(scan.csv_path),
                "best_gru_lr": float(best_result.lr),
                "n_test_windows_full": int(len(x_test)),
                "n_test_windows_sampled": int(len(eval_indices)),
                "test_mae_full": full_test_metrics["mae"],
                "test_rmse_full": full_test_metrics["rmse"],
                "test_r2_full": full_test_metrics["r2"],
                "test_mae_sampled": sampled_test_metrics["mae"],
                "test_rmse_sampled": sampled_test_metrics["rmse"],
                "test_r2_sampled": sampled_test_metrics["r2"],
            }
        )

        pd.DataFrame({"idx": eval_indices}).to_csv(
            dataset_dir / "selected_eval_indices.csv",
            index=False,
        )

        for idx in eval_indices[: cfg.example_plots_per_dataset]:
            plot_forecast_example(
                bundle=bundle,
                cfg=cfg,
                x_scaled=x_test[idx],
                y_true_scaled=y_test[idx],
                y_pred_scaled=y_pred[idx],
                start_idx=int(test_starts[idx]),
                title=f"{scan.name} | sampled forecast example {idx}",
                output_path=dataset_dir / f"forecast_example_{idx:04d}.png",
            )

        del (
            bundle,
            grid_results,
            best_result,
            x_test,
            y_test,
            y_pred,
            y_test_inv,
            y_pred_inv,
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    grid_search_df = pd.DataFrame(all_grid_rows).sort_values(["dataset", "gru_lr"])
    forecast_df = pd.DataFrame(all_forecast_rows).sort_values("dataset")

    save_dataframe(grid_search_df, output_dir / "grid_search_results.csv")
    save_dataframe(forecast_df, output_dir / "forecast_metrics.csv")

    plot_heatmap(
        grid_search_df.pivot(
            index="dataset",
            columns="gru_lr",
            values="val_mae",
        ).rename(columns=format_lr),
        title="Validation MAE across GRU learning rates",
        output_path=plots_root / "gru_lr_val_mae_heatmap.png",
    )
    plot_forecast_metrics_overview(
        forecast_df,
        plots_root / "forecast_metrics_overview.png",
    )

    print(f"\nAll GRU outputs saved to: {output_dir}")
    return output_dir


set_seed(cfg.seed)
run_output_dir = run_gru_model_selection(
    search_root=SEARCH_ROOT,
    output_dir=OUTPUT_DIR,
    cfg=cfg,
)
run_output_dir


## 6. Inspect the saved GRU results


In [ ]:
grid_search_results = pd.read_csv(run_output_dir / "grid_search_results.csv")
forecast_metrics = pd.read_csv(run_output_dir / "forecast_metrics.csv")

display(grid_search_results)
display(forecast_metrics)
